# 🌲 Cypress: GPU-Accelerated PCB Placement on Google Colab

This notebook runs the prebuilt **Cypress** Apptainer container (`cypress.sif`) on Google Colab's Free Tier NVIDIA T4 GPU.

### How It Works (Protocol Option A: Google Drive Batch Queue)
1. Mounts Google Drive (`/content/drive/MyDrive/Cypress_Colab`).
2. Installs Apptainer in seconds (unprivileged, no docker daemon required).
3. Passes the Colab T4 GPU to the container via `--nv`.
4. Starts the **Queue Worker**, which polls `queue/` on Google Drive, runs placement, and outputs final `.gp.pl` coordinates and plots to `completed/`.

## Step 1: Check GPU & Install Apptainer

In [ ]:
# Verify GPU allocation (Google Colab Free Tier typically gives an NVIDIA Tesla T4)
!nvidia-smi

# Install Apptainer via official PPA (runs in ~10 seconds)
!sudo apt-get update && sudo apt-get install -y software-properties-common
!sudo add-apt-repository -y ppa:apptainer/ppa
!sudo apt-get update -qq
!sudo apt-get install -y -qq apptainer

# Configure Apptainer for Colab container permissions
!sudo sed -i 's/^root default capabilities = full/root default capabilities = no/' /etc/apptainer/apptainer.conf
!sudo sed -i 's/^enable overlay = .*/enable overlay = driver/' /etc/apptainer/apptainer.conf
!sudo sed -i 's/^enable underlay = .*/enable underlay = preferred/' /etc/apptainer/apptainer.conf
!sudo sed -i 's/^max underlay binds = .*/max underlay binds = 5000/' /etc/apptainer/apptainer.conf

# Register Colab's NVIDIA driver libraries in ldconfig so Apptainer finds libcuda.so.1
!echo "/usr/lib64-nvidia" | sudo tee /etc/ld.so.conf.d/nvidia.conf
!sudo ldconfig

# Initialize host CUDA to ensure /dev/nvidia-uvm is instantiated
import torch
print("Host CUDA available:", torch.cuda.is_available(), "| Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
if torch.cuda.is_available():
    torch.zeros(1).cuda()

!apptainer version

## Step 2: Mount Google Drive & Define Paths

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount your Google Drive
drive.mount('/content/drive')

# Set the path to your shared workspace folder in Google Drive
candidates = [
    Path('/content/drive/MyDrive/cypress_colab_runtime'),
    Path('/content/drive/MyDrive/Cypress_Colab'),
]
WORKSPACE = next((p for p in candidates if (p / 'cypress.sif').exists()), candidates[0])
SIF_PATH = WORKSPACE / 'cypress.sif'

# Ensure required queue directories exist
for sub in ['queue', 'running', 'completed', 'failed']:
    (WORKSPACE / sub).mkdir(parents=True, exist_ok=True)

print(f"Workspace: {WORKSPACE}")
print(f"SIF Image Exists: {SIF_PATH.exists()}")
if not SIF_PATH.exists():
    print("\n[!] WARNING: cypress.sif not found! Please upload cypress.sif to Google Drive under 'Cypress_Colab/'.")

## Step 3: Smoke Test GPU Inside the Container

In [ ]:
# Verify that PyTorch inside the prebuilt container sees Colab's GPU
!apptainer exec --nv --no-privs \
  -B /usr/lib64-nvidia:/usr/lib64-nvidia \
  --env LD_LIBRARY_PATH="/usr/lib64-nvidia:/usr/local/nvidia/lib64" \
  "{SIF_PATH}" python3 -c \
  "import torch, dreamplace.ops.place_io.place_io as pio; print('PyTorch Version:', torch.__version__); print('CUDA Available:', torch.cuda.is_available()); print('Device Name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')"

## Step 4: Run the Queue Worker (Batch Mode)

Run this cell while you want Colab to process jobs. Whenever you drop a job into Google Drive's `queue/` folder, this cell will process it and move the result to `completed/`.

In [ ]:
# Start queue worker daemon with local patched worker script
import subprocess

worker_code = '''#!/usr/bin/env python3
import argparse, json, os, re, shutil, subprocess, sys, time
from datetime import datetime
from pathlib import Path

def parse_summary_from_log(log_path):
    summary = {"status": "UNKNOWN", "final_hpwl": None, "final_overflow": None, "iterations": None, "timestamp": datetime.now().isoformat()}
    if not os.path.exists(log_path): return summary
    with open(log_path, "r", errors="ignore") as f: content = f.read()
    matches = re.findall(r"iteration\\s+(\\d+).*?HPWL\\s+([\\d\\.E\\+\\-]+).*?Overflow\\s+([\\d\\.E\\+\\-]+)", content)
    if matches:
        last_iter, last_hpwl, last_overflow = matches[-1]
        summary["iterations"] = int(last_iter)
        try:
            summary["final_hpwl"] = float(last_hpwl)
            summary["final_overflow"] = float(last_overflow)
        except ValueError: pass
    if "placement succeeds" in content.lower() or summary["iterations"] is not None: summary["status"] = "SUCCESS"
    return summary

def process_job(job_dir, cypress_root):
    job_path = Path(job_dir)
    print(f"\\n[WORKER] ===== Processing Job: {job_path.name} =====")
    start_time = time.time()
    config_candidates = list(job_path.glob("*.json"))
    config_file = next((c for c in config_candidates if c.name.lower() == "config.json"), config_candidates[0] if config_candidates else None)
    if not config_file: raise FileNotFoundError(f"No .json config file found in {job_path}")
    with open(config_file, "r") as f: config = json.load(f)
    if "aux_input" in config:
        aux_candidate = job_path / config["aux_input"]
        if aux_candidate.exists(): config["aux_input"] = str(aux_candidate)
        else:
            aux_files = list(job_path.glob("*.aux"))
            if aux_files: config["aux_input"] = str(aux_files[0])
    result_dir = job_path / "results"
    result_dir.mkdir(parents=True, exist_ok=True)
    config["result_dir"] = str(result_dir)
    config["gpu"] = 1
    resolved_config = job_path / "resolved_config.json"
    with open(resolved_config, "w") as f: json.dump(config, f, indent=2)
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{cypress_root}:{env.get('PYTHONPATH', '')}".rstrip(":")
    cmd = [sys.executable, str(Path(cypress_root) / "dreamplace" / "Placer.py"), str(resolved_config)]
    log_path = job_path / "DREAMPlace.log"
    print(f"[WORKER] Executing command: {' '.join(cmd)}")
    with open(log_path, "w") as log_file:
        proc = subprocess.run(cmd, cwd=str(job_path), env=env, stdout=log_file, stderr=subprocess.STDOUT, text=True)
    elapsed = time.time() - start_time
    print(f"[WORKER] Completed in {elapsed:.2f}s. Exit code: {proc.returncode}")
    for artifact in result_dir.glob("*"):
        if artifact.is_file():
            dest = job_path / artifact.name
            if not dest.exists(): shutil.copy2(artifact, dest)
    summary = parse_summary_from_log(log_path)
    summary["elapsed_seconds"] = round(elapsed, 2)
    summary["exit_code"] = proc.returncode
    with open(job_path / "summary.json", "w") as sf: json.dump(summary, sf, indent=2)
    if proc.returncode != 0: raise RuntimeError(f"Cypress exited with code {proc.returncode}. See DREAMPlace.log")

def run_worker(workspace, cypress_root, poll_interval=5):
    workspace = Path(workspace).resolve()
    queue_dir, running_dir, completed_dir, failed_dir = workspace/"queue", workspace/"running", workspace/"completed", workspace/"failed"
    for d in [queue_dir, running_dir, completed_dir, failed_dir]: d.mkdir(parents=True, exist_ok=True)
    print("=" * 60)\n    print("  CYPRESS GOOGLE DRIVE QUEUE WORKER STARTED")\n    print(f"  Workspace:      {workspace}")\n    print(f"  Poll Interval:  {poll_interval}s")\n    print("=" * 60)
    print(f"[WORKER] Monitoring '{queue_dir}' for incoming jobs...")
    while True:
        try:
            job_dirs = sorted([p for p in queue_dir.iterdir() if p.is_dir()])
            if not job_dirs: time.sleep(poll_interval); continue
            for job in job_dirs:
                ready_marker = job / "READY"
                if (job / ".uploading").exists() or not ready_marker.exists(): continue
                if not list(job.glob("*.json")): continue
                job_name = job.name
                target_running, target_completed, target_failed = running_dir/job_name, completed_dir/job_name, failed_dir/job_name
                if target_running.exists(): shutil.rmtree(target_running)
                print(f"[WORKER] Claiming job {job_name} (READY detected)...")
                try: ready_marker.unlink()
                except OSError: pass
                shutil.move(str(job), str(target_running))
                try:
                    process_job(target_running, cypress_root)
                    if target_completed.exists(): shutil.rmtree(target_completed)
                    shutil.move(str(target_running), str(target_completed))
                    print(f"[WORKER] SUCCESS: Job {job_name} moved to {completed_dir}")
                except Exception as err:
                    print(f"[WORKER] ERROR processing {job_name}: {err}")
                    with open(target_running / "error.log", "a") as ef:
                        ef.write(f"Timestamp: {datetime.now().isoformat()}\\nError: {err}\\n")
                    if target_failed.exists(): shutil.rmtree(target_failed)
                    shutil.move(str(target_running), str(target_failed))
                    print(f"[WORKER] FAILED: Job {job_name} moved to {failed_dir}")
        except KeyboardInterrupt:
            print("\\n[WORKER] Stopped by user."); break
        except Exception as e:
            print(f"[WORKER] Loop error: {e}"); time.sleep(poll_interval)

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--workspace', type=str, default='/workspace')
    parser.add_argument('--cypress-root', type=str, default='/Cypress')
    parser.add_argument('--poll-interval', type=int, default=5)
    args = parser.parse_args()
    run_worker(args.workspace, args.cypress_root, args.poll_interval)
'''
with open('/content/colab_queue_worker.py', 'w') as f:
    f.write(worker_code)

!apptainer exec --nv --no-privs \
  -B /usr/lib64-nvidia:/usr/lib64-nvidia \
  -B "{WORKSPACE}:/workspace" \
  -B /content/colab_queue_worker.py:/colab_queue_worker.py \
  --env LD_LIBRARY_PATH="/usr/lib64-nvidia:/usr/local/nvidia/lib64" \
  "{SIF_PATH}" \
  python3 /colab_queue_worker.py --workspace /workspace --poll-interval 5

## (Optional) Step 5: Direct Single-Job Execution

If you prefer running a single benchmark interactively without the queue, run this cell:

In [ ]:
# Example: run the built-in benchmark directly
!apptainer exec --nv --no-privs \
  -B /usr/lib64-nvidia:/usr/lib64-nvidia \
  --env LD_LIBRARY_PATH="/usr/lib64-nvidia:/usr/local/nvidia/lib64" \
  "{SIF_PATH}" python3 /Cypress/dreamplace/Placer.py /Cypress/configs/test_small1.json